# PEDS — experimentos completos (curvas, tabelas, eficiência de dados, AL)

5 surrogates do paper. Curvas baseline vs PEDS, 2 tabelas da página 21, sweep de tamanho
(eficiência de dados) + tabela-veredito do "mais com menos", e active learning do Maxwell.
`N_ENSEMBLE=5` aplica a média de 5 modelos **nos dois lados** (PEDS e NN-only), como no paper.

In [ ]:
# Célula 1 — setup
%matplotlib inline
import os, sys, torch
sys.path.append(os.path.abspath('..'))
from src.peds_experiments import (EXPERIMENTS, run_diffusion_experiment,
                                   plot_learning_curve, plot_replication_tables,
                                   run_size_sweep, plot_size_sweep, plot_efficiency_table)
from src.maxwell_experiment import (run_maxwell_experiment,
                                     run_maxwell_active_learning, plot_al_curve)
DATA_ROOT = os.path.abspath('../data')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_ENSEMBLE = 1          # 1 = modelo único | 5 = ensemble do paper (PEDS e NN-only, treina 5x)
print('device:', DEVICE, '| ensemble:', N_ENSEMBLE)
results = {}

## Experimentos 1–4: difusão (Fourier) e reação-difusão (Fisher)

In [ ]:
# Célula 2 — Fourier(16/25), Fisher(16/25)
for name in EXPERIMENTS:
    print(f'=== {name} ===')
    r = run_diffusion_experiment(name, DATA_ROOT, device=DEVICE,
                                 telemetry=True, n_ensemble=N_ENSEMBLE)
    results[name] = r
    print(f"  PEDS={r['final_peds']:.3f} | NN-only={r['final_nn']:.3f} | "
          f"low-fi={r['lowfi']:.3f} | w={r['w']:.3f}")
    plot_learning_curve(r)

## Experimento 5: Maxwell(10) — treino estático (PEDS vs NN-only, ambos ensemble)

In [ ]:
# Célula 3 — Maxwell(10) estático
rm = run_maxwell_experiment(DATA_ROOT, device=DEVICE, telemetry=True,
                            grad_clip=1.0, n_ensemble=N_ENSEMBLE)
results['Maxwell(10)'] = rm
print(f"PEDS={rm['final_peds']:.3f} | NN-only={rm['final_nn']:.3f} | "
      f"low-fi={rm['lowfi']:.3f} | w={rm['w']:.3f}")
plot_learning_curve(rm)

## As 2 tabelas da página 21 (premissa de ACURÁCIA)
Tabela 2 = PEDS vs NN-only; Tabela 3 = PEDS vs low-fidelity. Mostram que o PEDS é mais
preciso com o MESMO (pouco) dado. A premissa de eficiência de dados ("menos dados") está
no sweep abaixo, não aqui.

In [ ]:
# Célula 4 — as 2 tabelas (replicado vs paper)
plot_replication_tables(results)

## Eficiência de dados — sweep de tamanho (premissa de "MENOS dados")
FE × nº de pontos (difusão). A tabela-veredito diz com quantos pontos cada modelo
atinge 5% e a economia de dados do PEDS. (Maxwell tem seu próprio gráfico na Célula 6.)

In [ ]:
# Célula 5 — sweep + tabela-veredito do 'mais com menos'
sweeps = []
for name in EXPERIMENTS:
    print(f'=== sweep {name} ===')
    sw = run_size_sweep(name, DATA_ROOT, sizes=[256,512,1024,2048,4096], device=DEVICE)
    sweeps.append(sw); plot_size_sweep(sw, target=0.05)
plot_efficiency_table(sweeps, target=0.05)

## Maxwell — eficiência de dados + active learning (NN-only vs PEDS vs PEDS+AL)
Um único gráfico com as 3 curvas em FE × nº de pontos (análogo da Fig. 3).
**Mais pesado** (ensemble × T × FDFD, em 3 estratégias). Reproduz a estratégia de aquisição
do paper dentro dos 10k pontos rotulados — sem o solver HF não alcança o trecho de 500k.

In [ ]:
# Célula 6 — Maxwell: NN-only vs PEDS vs PEDS+AL  (LENTO)
al = run_maxwell_active_learning(DATA_ROOT, device=DEVICE,
                                 n_ensemble=3, ninit=256, T=6, M=4, K=128, epochs=8,
                                 grad_clip=1.0, compare_random=True, compare_nn=True)
plot_al_curve(al, target=0.20)